# CoT_SFT

This notebook is a CoT-adapted version of the original SFT workflow.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
!pip install -q datasets trl peft accelerate
!pip install -q -U bitsandbytes>=0.46.1


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 50.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 56.9 MB/s eta 0:00:00


In [ ]:
from datasets import load_dataset, DatasetDict

DATA_ROOT = "/content/drive/MyDrive/CSE545/data"
RAW_COT_FILE = f"{DATA_ROOT}/train_cot_sft.json"

print("Loading CoT dataset from:", RAW_COT_FILE)
loaded = load_dataset("json", data_files=RAW_COT_FILE)

# load_dataset may return a DatasetDict; split must be called on a Dataset.
if isinstance(loaded, DatasetDict):
    raw_dataset = loaded["train"] if "train" in loaded else next(iter(loaded.values()))
else:
    raw_dataset = loaded

split_dataset = raw_dataset.train_test_split(test_size=0.1, seed=42, shuffle=True)
dataset = DatasetDict({
    "train": split_dataset["train"],
    "validation": split_dataset["test"],
})

print("Split complete")
print(f"Train size: {len(dataset['train'])}")
print(f"Validation size: {len(dataset['validation'])}")
print("Sample keys:", dataset["train"].column_names)

Loading CoT dataset from: /content/drive/MyDrive/CSE545/data/train_cot_sft.json


Generating train split: 0 examples [00:00, ? examples/s]

Split complete
Train size: 54595
Validation size: 6067
Sample keys: ['instruction', 'input', 'output', 'source']


In [ ]:
from transformers import AutoTokenizer

print("Loading tokenizer...")
model_id = "deepseek-ai/DeepSeek-R1-Distill-Llama-8B"
tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token


Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
print("Formatting prompts for completion-only SFT...")

def format_prompts(examples):
    instructions = examples["instruction"]
    inputs = examples["input"]
    outputs = examples["output"]

    prompts = []
    completions = []

    for instruction, input_text, output in zip(instructions, inputs, outputs):
        prompt = (
            f"### Instruction:\n{instruction}\n\n"
            f"### Input:\n{input_text}\n\n"
            f"### Response:\n"
        )
        completion = f"{output}{tokenizer.eos_token}"
        prompts.append(prompt)
        completions.append(completion)

    return {"prompt": prompts, "completion": completions}

processed_dataset = dataset.map(format_prompts, batched=True)
original_columns = dataset["train"].column_names
processed_dataset = processed_dataset.remove_columns(original_columns)

FORMATTED_TRAIN_FILE = f"{DATA_ROOT}/train_cot_dataset_formatted.jsonl"
FORMATTED_VAL_FILE = f"{DATA_ROOT}/val_cot_dataset_formatted.jsonl"
processed_dataset["train"].to_json(FORMATTED_TRAIN_FILE, lines=True, force_ascii=False)
processed_dataset["validation"].to_json(FORMATTED_VAL_FILE, lines=True, force_ascii=False)

print("Saved formatted files:")
print(FORMATTED_TRAIN_FILE)
print(FORMATTED_VAL_FILE)
print("Formatted sample:")
print(processed_dataset["train"][0])


Formatting prompts for completion-only SFT...


Map:   0%|          | 0/54595 [00:00<?, ? examples/s]

Map:   0%|          | 0/6067 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/55 [00:00<?, ?ba/s]

Creating json from Arrow format:   0%|          | 0/7 [00:00<?, ?ba/s]

Saved formatted files:
/content/drive/MyDrive/CSE545/data/train_cot_dataset_formatted.jsonl
/content/drive/MyDrive/CSE545/data/val_cot_dataset_formatted.jsonl
Formatted sample:
{'prompt': '### Instruction:\nYou are a financial analyst. Analyze the sentiment of the following financial text. Think step by step before giving your final answer. Answer with positive, negative, or neutral.\n\n### Input:\nVERY BULLISH...THIS NEW NEWS...\n\n\n### Response:\n', 'completion': '<think>\nThe text uses "VERY BULLISH" which is a strong, direct term in finance indicating a strong expectation that prices will rise. The capitalization and ellipsis add emphasis and excitement. The phrase "THIS NEW NEWS..." suggests that recent information has triggered this optimistic outlook, implying a positive catalyst for the market or asset in question.\n</think>\npositive<｜end▁of▁sentence｜>'}


# Training

Reload the processed JSONL files from Drive and start LoRA fine-tuning.


In [ ]:
import os
from datasets import load_dataset

if not os.path.exists(FORMATTED_TRAIN_FILE) or not os.path.exists(FORMATTED_VAL_FILE):
    raise FileNotFoundError("Formatted train/validation jsonl files are missing.")

print("Reloading formatted JSONL...")
processed_dataset = load_dataset(
    "json",
    data_files={
        "train": FORMATTED_TRAIN_FILE,
        "validation": FORMATTED_VAL_FILE,
    },
)

print(processed_dataset)
print("Formatted train sample:")
print(processed_dataset["train"][0])


Reloading formatted JSONL...
DatasetDict({
    train: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 54595
    })
    validation: Dataset({
        features: ['prompt', 'completion'],
        num_rows: 6067
    })
})
Formatted train sample:
{'prompt': '### Instruction:\nYou are a financial analyst. Analyze the sentiment of the following financial text. Think step by step before giving your final answer. Answer with positive, negative, or neutral.\n\n### Input:\nVERY BULLISH...THIS NEW NEWS...\n\n\n### Response:\n', 'completion': '<think>\nThe text uses "VERY BULLISH" which is a strong, direct term in finance indicating a strong expectation that prices will rise. The capitalization and ellipsis add emphasis and excitement. The phrase "THIS NEW NEWS..." suggests that recent information has triggered this optimistic outlook, implying a positive catalyst for the market or asset in question.\n</think>\npositive<｜end▁of▁sentence｜>'}


In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

MODEL_ROOT = "/content/drive/MyDrive/CSE545/models"
RUN_NAME = "deepseek-financial-cot-lora"
FINAL_ADAPTER_DIR = f"{MODEL_ROOT}/{RUN_NAME}-final"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

training_args = SFTConfig(
    output_dir=f"{MODEL_ROOT}/{RUN_NAME}",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    optim="paged_adamw_32bit",
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    save_strategy="steps",
    save_steps=100,
    save_total_limit=3,
    logging_steps=10,
    num_train_epochs=3,
    max_steps=-1,
    fp16=False,
    bf16=True,
    max_length=2048,
    completion_only_loss=True,
    report_to="none",
)

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

print("Training config ready")
print("Model output dir:", training_args.output_dir)
print("Final adapter dir:", FINAL_ADAPTER_DIR)
print("Checkpoint cadence: every", training_args.save_steps, "steps")


Training config ready
Model output dir: /content/drive/MyDrive/CSE545/models/deepseek-financial-cot-lora
Final adapter dir: /content/drive/MyDrive/CSE545/models/deepseek-financial-cot-lora-final
Checkpoint cadence: every 100 steps


In [ ]:
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
)
model.config.use_cache = False
model.gradient_checkpointing_enable()
model = prepare_model_for_kbit_training(model)

print("Model loaded and prepared")


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Model loaded and prepared


In [ ]:
print("Initializing SFTTrainer...")
trainer = SFTTrainer(
    model=model,
    train_dataset=processed_dataset["train"],
    eval_dataset=processed_dataset["validation"],
    peft_config=peft_config,
    processing_class=tokenizer,
    args=training_args,
)
print("SFTTrainer ready")


Initializing SFTTrainer...
SFTTrainer ready


add checkpoint to avoid interrupt of google colab

In [ ]:
import os
import re


def get_latest_checkpoint(output_dir):
    if not os.path.isdir(output_dir):
        return None

    checkpoints = []
    for name in os.listdir(output_dir):
        match = re.fullmatch(r"checkpoint-(\d+)", name)
        if match:
            checkpoints.append((int(match.group(1)), os.path.join(output_dir, name)))

    if not checkpoints:
        return None

    checkpoints.sort(key=lambda x: x[0])
    return checkpoints[-1][1]


latest_checkpoint = get_latest_checkpoint(training_args.output_dir)
if latest_checkpoint:
    print("Resuming from checkpoint:", latest_checkpoint)
else:
    print("Starting CoT fine-tuning from scratch...")

trainer.train(resume_from_checkpoint=latest_checkpoint)

# Save final adapter snapshot in a stable Drive path.
trainer.model.save_pretrained(FINAL_ADAPTER_DIR)
print("Training complete. Adapter saved to:", FINAL_ADAPTER_DIR)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 128001}.


Starting CoT fine-tuning from scratch...


Step,Training Loss
10,1.586102
20,1.273971
30,1.224956
40,1.149857
50,1.161995
